In [1]:

path = '/home/apolo/Documents/github_projects/ripple_sync/'

scripts_path = path + '/scripts/'
tables_path = path + '/tables/'
figures_path = path + '/figures/'


In [2]:

import pandas as pd
import os 
import numpy as np
import matplotlib.pyplot as plt
import scipy
import seaborn as sns
import dabest

from statsmodels.stats.proportion import proportions_chisquare

%matplotlib widget

import helper_functions as hf


In [3]:

import warnings

warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")


In [4]:

import statsmodels.api as sm
import statsmodels.formula.api as smf


In [5]:

# set the font globally
plt.rcParams.update({'font.family':'sans-serif'})
# set the font name for a font family
plt.rcParams.update({'font.sans-serif':'Arial'})
# when saving svg, keep text as text
plt.rcParams['svg.fonttype'] = 'none' 


In [6]:
import numpy as np
from scipy.stats import t
from statsmodels.stats.power import TTestIndPower


def calculate_mde(n1, n2, alpha=0.05, power=0.8):
    """
    Calculate the Minimum Detectable Effect (MDE) for a given study design.

    Parameters:
    - n1, n2: Sample sizes for group1 and group2.
    - alpha: Significance level (default 0.05).
    - power: Desired statistical power (default 0.8).

    Returns:
    - Minimum detectable effect size (Cohen's d).
    """
    analysis = TTestIndPower()
    mde = analysis.solve_power(effect_size=None, nobs1=n1, ratio=n2/n1, alpha=alpha, power=power, alternative='two-sided')
    return mde


def equivalence_test(group1, group2, sesoi):
    """
    Calculate t-values for TOST procedure with equal variances.

    Parameters:
    - group1: Array-like, data for group 1.
    - group2: Array-like, data for group 2.
    - lower_bound: Lower equivalence bound (in raw score units).
    - upper_bound: Upper equivalence bound (in raw score units).

    Returns:
    - t_lower: t-value for the lower bound test.
    - t_upper: t-value for the upper bound test.
    """
    # Sample sizes
    n1, n2 = len(group1), len(group2)

    # Sample means
    mean1, mean2 = np.nanmean(group1), np.nanmean(group2)

    # Sample standard deviations
    std1, std2 = np.nanstd(group1, ddof=1), np.nanstd(group2, ddof=1)

    # Mean difference
    mean_diff = mean1 - mean2

    # pooled standard deviation
    sp = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))

    se_diff = sp * np.sqrt(1 / n1 + 1 / n2)  
    dof = n1 + n2 - 2

    # # Calculate dof
    # if n1 == n2:
    #     se_diff = sp * np.sqrt(1 / n1 + 1 / n2)  
    #     dof = n1 + n2 - 2
    # else:
    #     se_diff = np.sqrt((std1**2/ n1) + (std2**2 / n2))
    #     dof_up = (std1**2/n1 + std2**2/n2)**2
    #     dof_down = ((std1**2/n1)**2)/(n1-1) + ((std2**2/n2)**2)/(n2-1)
    #     dof = dof_up/dof_down
        
    # Equivalence bounds
    delta_lower_bound = -sesoi * sp
    delta_upper_bound = sesoi * sp

    # Calculate t-values for the bounds
    t_lower = (mean_diff - delta_lower_bound) / se_diff
    t_upper = (mean_diff - delta_upper_bound) / se_diff

    # Calculate p-values for the bounds
    p_lower = 1 - t.cdf(t_lower, dof)  # One-tailed lower
    p_upper = t.cdf(t_upper, dof)  # One-tailed upper

    p_value = np.max([p_lower,p_upper])

    return  {
        "mean_diff": mean_diff,
        "t_lower": t_lower,
        "t_upper": t_upper,
        "delta_lower_bound": delta_lower_bound,
        "delta_upper_bound": delta_upper_bound,
        "p_lower": p_lower,
        "p_upper": p_upper,
        "p_value": p_value
    }



In [7]:
os.chdir(tables_path + '/figure_2/')

filename = 'Figure_Ripple_Abundance_Overall.csv'
df = pd.read_csv(filename)

shank_mask = (df['shank_distance'] == 0.0) | (df['shank_distance'] == 400.0)
side_mask = df['shank_side'] == 'right'
rat_mask = df['rat'] == 'Bud'
df = df[~(shank_mask*side_mask*rat_mask)]

# Do this to use in the linear mixed model formula with varying intercepts
# (i.e., 1|session, meaning that each session contains its own baseline, similar to paired data)
df['session'] = pd.Categorical(df['session'])
df['rat'] = pd.Categorical(df['rat'])
df = df.reset_index(drop=True)
df

,rat,session,shank_side,shank_distance,ripple_abundance
0,Ach,Achilles_10252013,left,0.0,0.500833
1,Ach,Achilles_10252013,left,200.0,0.500000
2,Ach,Achilles_10252013,left,400.0,0.488611
3,Ach,Achilles_10252013,left,600.0,0.481111
4,Ach,Achilles_10252013,left,800.0,0.479167
...,...,...,...,...,...
85,Gat,Gatsby_08282013,left,1200.0,0.485556
86,Gat,Gatsby_08282013,right,800.0,0.588056
87,Gat,Gatsby_08282013,right,1000.0,0.589444
88,Gat,Gatsby_08282013,right,1200.0,0.622500


In [8]:

ripple_abundance_left = np.array(df.ripple_abundance[df.shank_side == 'left'])
ripple_abundance_right = np.array(df.ripple_abundance[df.shank_side == 'right'])


In [9]:
n1 = ripple_abundance_left.shape[0]
n2 = ripple_abundance_right.shape[0]

mde = calculate_mde(n1, n2, alpha=0.05, power=0.8)
mde

0.6008896445268586

In [10]:
sesoi = 0.6

results = equivalence_test(ripple_abundance_left, ripple_abundance_right, sesoi)
results


{'mean_diff': -0.00454583333333336,
 't_lower': 2.437110233337206,
 't_upper': -3.219744016155174,
 'delta_lower_bound': -0.03285715129311251,
 'delta_upper_bound': 0.03285715129311251,
 'p_lower': 0.008409015582118706,
 'p_upper': 0.0008989557159563155,
 'p_value': 0.008409015582118706}

In [11]:

sessions = ['Achilles_10252013', 
            'Achilles_11012013',
            'Buddy_06272013',
            'Cicero_09012014',
            'Cicero_09102014',
            'Cicero_09172014',
            'Gatsby_08022013',
            'Gatsby_08282013']

os.chdir(tables_path + '/figure_2/')

df = pd.DataFrame()
ripple_freq_left = []
ripple_freq_right = []
for session in sessions:
    
    filename = 'Figure_Ripple_Frequency_Distribution_' + session + '.npy'
    df_pre = np.load(filename,allow_pickle=True).item()
    df_pre = pd.DataFrame(df_pre)
        
    ripple_mean_frequency = []
    for ii in range(df_pre['ripple_frequency'].shape[0]):
        ripple_mean_frequency.append(np.nanmean(df_pre['ripple_frequency'][ii]))
    df_pre['ripple_frequency'] = ripple_mean_frequency
    df = pd.concat([df,df_pre])

df = df.rename(columns={'ripple_frequency': 'ripple_mean_frequency'})


shank_mask = (df['shank_distance'] == 0.0) | (df['shank_distance'] == 400.0)
side_mask = df['shank_side'] == 'right'
rat_mask = df['rat'] == 'Bud'
df = df[~(shank_mask*side_mask*rat_mask)]

# # Do this to use in the linear mixed model formula with varying intercepts
# # (i.e., 1|session, meaning that each session contains its own baseline, similar to paired data)
df['session'] = pd.Categorical(df['session'])
df['rat'] = pd.Categorical(df['rat'])
df = df.reset_index(drop=True)
df

,rat,session,shank_side,shank_distance,ripple_mean_frequency
0,Ach,Achilles_10252013,left,0.0,151.859124
1,Ach,Achilles_10252013,left,200.0,151.642222
2,Ach,Achilles_10252013,left,400.0,152.494599
3,Ach,Achilles_10252013,left,600.0,153.863741
4,Ach,Achilles_10252013,left,800.0,154.871884
...,...,...,...,...,...
85,Gat,Gatsby_08282013,left,1200.0,140.310069
86,Gat,Gatsby_08282013,right,800.0,133.783656
87,Gat,Gatsby_08282013,right,1000.0,134.951932
88,Gat,Gatsby_08282013,right,1200.0,133.750112


In [12]:
ripple_freq_left = np.array(df.ripple_mean_frequency[df.shank_side == 'left'])
ripple_freq_right = np.array(df.ripple_mean_frequency[df.shank_side == 'right'])


In [13]:
n1 = ripple_freq_left.shape[0]
n2 = ripple_freq_right.shape[0]

mde = calculate_mde(n1, n2, alpha=0.05, power=0.8)
mde

0.6008896445268586

In [14]:
sesoi = 0.6

results = equivalence_test(ripple_freq_left, ripple_freq_right, sesoi)
results

{'mean_diff': -0.6134827749547753,
 't_lower': 2.4726181359023927,
 't_upper': -3.184236113589988,
 'delta_lower_bound': -4.876749535994466,
 'delta_upper_bound': 4.876749535994466,
 'p_lower': 0.007667745116970215,
 'p_upper': 0.0010037452425846972,
 'p_value': 0.007667745116970215}

In [15]:

import pingouin as pg

tost_stats = pg.tost(ripple_freq_left, ripple_freq_right,bound=4)
tost_stats


,bound,dof,pval
TOST,4,88,0.026336


In [16]:
os.chdir(tables_path + '/figure_2/')

filename = 'Figure_Ripple_InterRippleInterval_OverallMean.csv'
df = pd.read_csv(filename)
df

shank_mask = (df['shank_distance'] == 0.0) | (df['shank_distance'] == 400.0)
side_mask = df['shank_side'] == 'right'
rat_mask = df['rat'] == 'Bud'
df = df[~(shank_mask*side_mask*rat_mask)]

# Do this to use in the linear mixed model formula with varying intercepts
# (i.e., 1|session, meaning that each session contains its own baseline, similar to paired data)
df['session'] = pd.Categorical(df['session'])
df['rat'] = pd.Categorical(df['rat'])
df = df.reset_index(drop=True)
df

,rat,session,shank_side,shank_distance,ripple_iri
0,Ach,Achilles_10252013,left,0.0,1.994846
1,Ach,Achilles_10252013,left,200.0,2.000216
2,Ach,Achilles_10252013,left,400.0,2.044720
3,Ach,Achilles_10252013,left,600.0,2.078295
4,Ach,Achilles_10252013,left,800.0,2.086729
...,...,...,...,...,...
85,Gat,Gatsby_08282013,left,1200.0,2.056022
86,Gat,Gatsby_08282013,right,800.0,1.699473
87,Gat,Gatsby_08282013,right,1000.0,1.695463
88,Gat,Gatsby_08282013,right,1200.0,1.606214


In [17]:
ripple_iri_left = np.array(df.ripple_iri[df.shank_side == 'left'])
ripple_iri_right = np.array(df.ripple_iri[df.shank_side == 'right'])

import pingouin as pg

tost_stats = pg.tost(ripple_iri_left, ripple_iri_right,bound=0.1)
tost_stats


,bound,dof,pval
TOST,0.1,88,0.035407


In [18]:
n1 = ripple_iri_left.shape[0]
n2 = ripple_iri_right.shape[0]

mde = calculate_mde(n1, n2, alpha=0.05, power=0.8)
mde

0.6008896445268586

In [19]:
sesoi = 0.6

results = equivalence_test(ripple_iri_left, ripple_iri_right, sesoi)
results

{'mean_diff': 0.022880263361930497,
 't_lower': 3.3710107168856545,
 't_upper': -2.285843532606725,
 'delta_lower_bound': -0.11927223464137923,
 'delta_upper_bound': 0.11927223464137923,
 'p_lower': 0.0005570375113880255,
 'p_upper': 0.012332748365325594,
 'p_value': 0.012332748365325594}

In [20]:
os.chdir(tables_path + '/figure_2/')

filename = 'Figure_Ripple_NumberOfCycles.csv'
df = pd.read_csv(filename)

shank_mask = (df['shank_distance'] == 0.0) | (df['shank_distance'] == 400.0)
side_mask = df['shank_side'] == 'right'
rat_mask = df['rat'] == 'Bud'
df = df[~(shank_mask*side_mask*rat_mask)]


# Do this to use in the linear mixed model formula with varying intercepts
# (i.e., 1|session, meaning that each session contains its own baseline, similar to paired data)
df['session'] = pd.Categorical(df['session'])
df['rat'] = pd.Categorical(df['rat'])
df = df.reset_index(drop=True)
df

,rat,session,shank_side,shank_distance,ripple_cycles
0,Ach,Achilles_10252013,left,0.0,7.051026
1,Ach,Achilles_10252013,left,200.0,6.968333
2,Ach,Achilles_10252013,left,400.0,7.031836
3,Ach,Achilles_10252013,left,600.0,7.175520
4,Ach,Achilles_10252013,left,800.0,7.153623
...,...,...,...,...,...
85,Gat,Gatsby_08282013,left,1200.0,6.231121
86,Gat,Gatsby_08282013,right,800.0,6.341521
87,Gat,Gatsby_08282013,right,1000.0,6.426956
88,Gat,Gatsby_08282013,right,1200.0,6.486390


In [21]:
ripple_cycles_left = np.array(df.ripple_cycles[df.shank_side == 'left'])
ripple_cycles_right = np.array(df.ripple_cycles[df.shank_side == 'right'])

import pingouin as pg

tost_stats = pg.tost(ripple_cycles_left, ripple_cycles_right,bound=1)
tost_stats

,bound,dof,pval
TOST,1,88,2.575825e-14


In [22]:
n1 = ripple_cycles_left.shape[0]
n2 = ripple_cycles_right.shape[0]

mde = calculate_mde(n1, n2, alpha=0.05, power=0.8)
mde

0.6008896445268586

In [24]:
sesoi = 0.6

results = equivalence_test(ripple_cycles_left, ripple_cycles_right, sesoi)
results

{'mean_diff': -0.11003598525414215,
 't_lower': 1.721582257533268,
 't_upper': -3.9352719919591115,
 'delta_lower_bound': -0.2811855343149158,
 'delta_upper_bound': 0.2811855343149158,
 'p_lower': 0.04432942396364603,
 'p_upper': 8.282791477741862e-05,
 'p_value': 0.04432942396364603}